# Route Resilience — road segmentation (mit_b3 + SCSE U-Net, EMA, Lovász)

A maximal-quality DeepGlobe training recipe that keeps the project's `p1_segment`
API and adds the levers that reliably raise road-segmentation IoU on a 16 GB GPU:

- **SegFormer MiT-B3 encoder + U-Net decoder with SCSE attention** (channel +
  spatial gating in every decoder block; verified to build with the MiT encoder)
- **ComboLoss = BCE + soft-Dice + Lovász-hinge (+ soft-clDice ramped in late)** —
  Lovász is a direct IoU surrogate; clDice rewards road connectivity
- **EMA (exponential moving average) of the weights** evaluated/exported — a
  reliable, low-variance accuracy gain over the raw last-step weights
- road-aware crop sampling, geometric/colour/occlusion augmentation
- mixed precision, channels-last, gradient accumulation + clipping, discriminative
  encoder/decoder LRs, warmup + cosine decay
- **full-resolution overlapping (Hann-blended) validation** with global IoU
  threshold search, **flip + multi-scale TTA**, and an **occlusion-aware deploy
  threshold**; best/last resumable checkpoints
- the deployable checkpoint stores `encoder`, `arch`, `decoder_attention_type`
  and `threshold`, so `predict.py` / `load_checkpoint` rebuild and deploy it as-is

## Kaggle checklist
1. **+ Add Data** → the `balraj98` **DeepGlobe Road Extraction Dataset**.
2. Turn on a **GPU** (16 GB T4/P100 class) and **Internet** (first run, for pip +
   repo + ImageNet weights). For OOM: set `encoder` → `mit_b2`, keep 512 px.
3. **Run all**, then **Save Version** to retain `/kaggle/working/road_segmentation`.

In [ ]:
# 1. Reproducible environment and pinned dependencies.
import importlib
import importlib.util
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

REQUIRED = {
    "segmentation_models_pytorch": "segmentation-models-pytorch==0.3.4",
    "albumentations": "albumentations==2.0.8",
    "albucore": "albucore==0.0.24",
    "cv2": "opencv-python-headless==4.10.0.84",
}
PINNED_VERSIONS = {
    "segmentation-models-pytorch": "0.3.4",
    "albumentations": "2.0.8",
    "albucore": "0.0.24",
}
required_install = any(
    importlib.util.find_spec(m) is None for m in REQUIRED
)
if not required_install:
    for dist, expected in PINNED_VERSIONS.items():
        try:
            if version(dist) != expected:
                required_install = True
                break
        except PackageNotFoundError:
            required_install = True
            break

WHEEL_DIR = None  # set to an attached Kaggle Dataset of wheels for offline runs
if required_install:
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if WHEEL_DIR:
        cmd += ["--no-index", "--find-links", WHEEL_DIR]
    cmd += list(REQUIRED.values())
    subprocess.check_call(cmd)

import albumentations as A
import cv2
import numpy as np
import segmentation_models_pytorch as smp
import torch

print("torch:", torch.__version__, "| albumentations:", A.__version__, "| smp:", smp.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| memory (GiB):",
          round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
# 2. Load the project source. The notebook only uses the public P1 API.
import os
import shutil
from pathlib import Path

def detect_env():
    if os.path.exists("/kaggle") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if "google.colab" in sys.modules or os.path.exists("/content"):
        return "colab"
    return "local"

ENV = detect_env()
REPO_URL = "https://github.com/Akshat-Tiwari69/Trace.git"
REPO_BRANCH = "dev"
REPO_DIR = "/kaggle/working/Trace" if ENV == "kaggle" else "/content/Trace" if ENV == "colab" else "."

OFFLINE_REPO_DIR = None  # set to an attached source snapshot for offline runs
repo_path = Path(REPO_DIR)
if not (repo_path / "src" / "pipeline" / "p1_segment").exists():
    if OFFLINE_REPO_DIR:
        shutil.copytree(OFFLINE_REPO_DIR, repo_path, dirs_exist_ok=True)
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
                               REPO_URL, str(repo_path)])
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

from src.pipeline.p1_segment.model import build_model
from src.pipeline.p1_segment.losses import ComboLoss
print("environment:", ENV, "| source:", repo_path.resolve())

In [ ]:
# 3. Configuration, deterministic split, and input discovery.
import gc
import math
import random
from collections import defaultdict

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except AttributeError:
    pass

CONFIG = {
    # Architecture / resolution. mit_b3 + SCSE U-Net @512 is the strongest
    # reliable default on a 16 GB GPU. For OOM: encoder -> mit_b2 (keep 512).
    "encoder": "mit_b3",
    "arch": "unet",
    "decoder_attention_type": "scse",   # channel+spatial attention in the decoder
    "image_size": 512,
    "batch_size": 2,
    "grad_accum": 4,                     # effective batch 8 without a memory spike

    # Training budget (same order as a single Kaggle session).
    "epochs": 30,
    "warmup_epochs": 2,
    "base_lr": 2.0e-4,
    "encoder_lr_scale": 0.35,
    "weight_decay": 1.0e-4,
    "min_lr_ratio": 0.03,
    "max_grad_norm": 1.0,
    "ema_decay": 0.9998,                 # weights actually evaluated + exported

    # Loss mixture. Lovász targets IoU directly; clDice ramps in for connectivity.
    "bce_weight": 0.40,
    "dice_weight": 0.40,
    "lovasz_weight": 0.20,
    "cldice_weight": 0.10,
    "topology_epochs": 6,                # last N epochs that switch clDice on

    # Data / crop balance — roads are sparse, so mostly request road-bearing crops.
    "crops_per_image": 1,
    "positive_crop_probability": 0.72,
    "min_positive_fraction": 0.002,
    "max_crop_attempts": 12,
    "occlusion_probability": 0.35,
    "val_fraction": 0.10,
    "num_workers": min(4, os.cpu_count() or 2),

    # Whole-image overlap validation (not a centre crop).
    "val_stride": 384,                   # 25% overlap for 512 px windows
    "val_batch_size": 4,
    "validate_every": 2,
    "val_flip_tta": False,               # fast during training
    "final_flip_tta": True,              # H/V/both flips at the end
    "final_tta_scales": [1.0, 1.25],     # multi-scale at the final eval
    "thresholds": [round(x, 2) for x in np.arange(0.20, 0.71, 0.02)],
    "seed": SEED,
}

OUTPUT_DIR = Path("/kaggle/working/road_segmentation") if ENV == "kaggle" else repo_path / "models" / "road_segmentation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TAG = f"deepglobe_{CONFIG['encoder']}_scse_{CONFIG['image_size']}px"
BEST_PATH = OUTPUT_DIR / f"{TAG}_best.pt"
LAST_PATH = OUTPUT_DIR / f"{TAG}_last.pt"
HISTORY_PATH = OUTPUT_DIR / "history.csv"

def paired_images_in(folder):
    folder = Path(folder)
    pairs = []
    for image_path in folder.glob("*_sat.jpg"):
        mask_path = image_path.with_name(image_path.name.replace("_sat.jpg", "_mask.png"))
        if mask_path.exists():
            pairs.append((image_path, mask_path))
    return sorted(pairs)

def find_deepglobe_train_dir():
    roots = [Path("/kaggle/input")] if ENV == "kaggle" else (
        [Path("/content")] if ENV == "colab" else [Path("data/raw/deepglobe"), Path("data")])
    candidates = set()
    for root in roots:
        if root.exists():
            for mask_path in root.rglob("*_mask.png"):
                candidates.add(mask_path.parent)
    scored = [(len(paired_images_in(f)), f) for f in candidates]
    scored = [(n, f) for n, f in scored if n]
    if not scored:
        raise FileNotFoundError("No DeepGlobe *_sat.jpg / *_mask.png pairs found. Add the dataset with + Add Data.")
    return max(scored, key=lambda item: item[0])[1]

DATA_ROOT = find_deepglobe_train_dir()
all_pairs = paired_images_in(DATA_ROOT)
print("DATA_ROOT:", DATA_ROOT, "| labelled images:", len(all_pairs))
assert len(all_pairs) >= 20, "Unexpectedly small labelled dataset; verify DATA_ROOT."

def road_fraction(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Could not read mask: {mask_path}")
    return float((mask > 127).mean())

# Coverage-stratified holdout: balanced road density in train/val.
coverages = np.array([road_fraction(m) for _, m in all_pairs])
edges = np.unique(np.quantile(coverages, np.linspace(0, 1, 6)))
bin_ids = np.digitize(coverages, edges[1:-1], right=True)
rng = np.random.default_rng(SEED)
val_indices = []
for bin_id in np.unique(bin_ids):
    idx = np.flatnonzero(bin_ids == bin_id)
    rng.shuffle(idx)
    val_indices.extend(idx[:max(1, round(len(idx) * CONFIG["val_fraction"]))].tolist())
val_indices = set(val_indices)
train_pairs = [p for i, p in enumerate(all_pairs) if i not in val_indices]
val_pairs = [p for i, p in enumerate(all_pairs) if i in val_indices]
print(f"train: {len(train_pairs)} | validation: {len(val_pairs)} | "
      f"road coverage median: {np.median(coverages):.3%}")

In [ ]:
# 4. Road-aware training dataset and augmentation.
from torch.utils.data import DataLoader, Dataset
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class RoadAwareCropDataset(Dataset):
    def __init__(self, pairs, size, crops_per_image, positive_probability,
                 min_positive_fraction, max_attempts, transform):
        self.pairs = pairs; self.size = size; self.crops_per_image = crops_per_image
        self.positive_probability = positive_probability
        self.min_positive_fraction = min_positive_fraction
        self.max_attempts = max_attempts; self.transform = transform

    def __len__(self):
        return len(self.pairs) * self.crops_per_image

    @staticmethod
    def _read(image_path, mask_path):
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if image is None or mask is None:
            raise ValueError(f"Unreadable pair: {image_path}, {mask_path}")
        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB), (mask > 127).astype(np.uint8)

    def _crop(self, image, mask):
        h, w = mask.shape
        if h < self.size or w < self.size:
            ph, pw = max(0, self.size - h), max(0, self.size - w)
            image = cv2.copyMakeBorder(image, 0, ph, 0, pw, cv2.BORDER_REFLECT_101)
            mask = cv2.copyMakeBorder(mask, 0, ph, 0, pw, cv2.BORDER_CONSTANT, value=0)
            h, w = mask.shape
        wants_road = np.random.random() < self.positive_probability and mask.any()
        chosen = (0, 0)
        for _ in range(self.max_attempts):
            y = np.random.randint(0, h - self.size + 1)
            x = np.random.randint(0, w - self.size + 1)
            chosen = (y, x)
            if not wants_road or mask[y:y + self.size, x:x + self.size].mean() >= self.min_positive_fraction:
                break
        y, x = chosen
        return image[y:y + self.size, x:x + self.size], mask[y:y + self.size, x:x + self.size]

    def __getitem__(self, idx):
        image, mask = self._read(*self.pairs[idx % len(self.pairs)])
        image, mask = self._crop(image, mask)
        out = self.transform(image=image, mask=mask)
        return out["image"], out["mask"].unsqueeze(0).float()

S = CONFIG["image_size"]
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=20,
                       border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.40),
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=12, val_shift_limit=8, p=0.25),
    A.OneOf([A.GaussianBlur(blur_limit=(3, 5)), A.MotionBlur(blur_limit=5),
             A.GaussNoise()], p=0.20),
    A.CLAHE(clip_limit=2.0, p=0.10),
    # Blacked-out regions model tree/vehicle/building occlusions; keep the label.
    A.CoarseDropout(num_holes_range=(1, 5),
                    hole_height_range=(max(4, S // 32), max(8, S // 9)),
                    hole_width_range=(max(4, S // 32), max(8, S // 9)),
                    fill=0, fill_mask=None, p=CONFIG["occlusion_probability"]),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

train_ds = RoadAwareCropDataset(
    train_pairs, S, CONFIG["crops_per_image"], CONFIG["positive_crop_probability"],
    CONFIG["min_positive_fraction"], CONFIG["max_crop_attempts"], train_transform)

def seed_worker(_):
    s = torch.initial_seed() % (2**32); random.seed(s); np.random.seed(s)

loader_kwargs = dict(batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True,
                     num_workers=CONFIG["num_workers"], pin_memory=torch.cuda.is_available(),
                     worker_init_fn=seed_worker)
if CONFIG["num_workers"] > 0:
    loader_kwargs.update(persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_ds, **loader_kwargs)
print("training crops / epoch:", len(train_ds),
      "| updates / epoch:", math.ceil(len(train_loader) / CONFIG["grad_accum"]))

In [ ]:
# 5. Whole-image overlap (Hann) validation with flip + multi-scale TTA.
@torch.inference_mode()
def _flip_tta_prob(net, x, use_flip):
    with torch.amp.autocast(device_type="cuda", enabled=(device == "cuda")):
        probs = torch.sigmoid(net(x))
        if use_flip:
            for dims in ((3,), (2,), (2, 3)):
                probs = probs + torch.flip(torch.sigmoid(net(torch.flip(x, dims=dims))), dims=dims)
            probs = probs / 4.0
    return probs

def _starts(length, tile, stride):
    if length <= tile:
        return [0]
    starts = list(range(0, length - tile + 1, stride))
    if starts[-1] != length - tile:
        starts.append(length - tile)
    return starts

@torch.inference_mode()
def sliding_window_prob(net, image, tile, stride, batch_size, use_flip=False):
    h, w = image.shape[:2]
    hp, wp = max(h, tile), max(w, tile)
    padded = cv2.copyMakeBorder(image, 0, hp - h, 0, wp - w, cv2.BORDER_REFLECT_101)
    ys, xs = _starts(hp, tile, stride), _starts(wp, tile, stride)
    # Non-zero Hann weights erase seams while still covering the borders.
    weight = (np.outer(np.hanning(tile), np.hanning(tile)) + 0.05).astype(np.float32)
    total = np.zeros((hp, wp), np.float32); norm = np.zeros((hp, wp), np.float32)
    coords = [(y, x) for y in ys for x in xs]
    mean_t = torch.tensor(IMAGENET_MEAN); std_t = torch.tensor(IMAGENET_STD)
    for off in range(0, len(coords), batch_size):
        batch = coords[off:off + batch_size]
        tiles = np.stack([padded[y:y + tile, x:x + tile] for y, x in batch])
        t = torch.from_numpy(tiles).float().div_(255.0).sub_(mean_t).div_(std_t)
        t = t.permute(0, 3, 1, 2).contiguous(memory_format=torch.channels_last).to(device, non_blocking=True)
        out = _flip_tta_prob(net, t, use_flip)[:, 0].float().cpu().numpy()
        for (y, x), prob in zip(batch, out):
            total[y:y + tile, x:x + tile] += prob * weight
            norm[y:y + tile, x:x + tile] += weight
    return (total / np.maximum(norm, 1e-6))[:h, :w]

@torch.inference_mode()
def predict_prob(net, image, use_flip=False, scales=(1.0,)):
    """Probability map with optional flip + multi-scale TTA, at native size."""
    h, w = image.shape[:2]
    acc = np.zeros((h, w), np.float32)
    for s in scales:
        img_s = image if s == 1.0 else cv2.resize(image, (max(1, round(w * s)), max(1, round(h * s))),
                                                  interpolation=cv2.INTER_LINEAR)
        prob = sliding_window_prob(net, img_s, CONFIG["image_size"], CONFIG["val_stride"],
                                   CONFIG["val_batch_size"], use_flip=use_flip)
        if s != 1.0:
            prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
        acc += prob
    return acc / len(scales)

def evaluate_full_images(net, pairs, thresholds, use_flip=False, scales=(1.0,), progress=True):
    inter = {float(t): 0 for t in thresholds}
    union = {float(t): 0 for t in thresholds}
    iterator = pairs
    if progress:
        from tqdm.auto import tqdm
        iterator = tqdm(pairs, desc="full-resolution validation", leave=False)
    net.eval()
    for image_path, mask_path in iterator:
        image = cv2.cvtColor(cv2.imread(str(image_path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        target = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127
        prob = predict_prob(net, image, use_flip=use_flip, scales=scales)
        for t in thresholds:
            pred = prob >= t
            inter[float(t)] += np.logical_and(pred, target).sum(dtype=np.int64)
            union[float(t)] += np.logical_or(pred, target).sum(dtype=np.int64)
    ious = {t: inter[t] / max(union[t], 1) for t in inter}
    best = max(ious, key=ious.get)
    return {"iou": float(ious[best]), "threshold": float(best), "iou_by_threshold": ious}

In [ ]:
# 6. Model (SCSE U-Net) + EMA + discriminative optimizer + schedule + ComboLoss.
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise RuntimeError("Enable a Kaggle GPU accelerator before training this configuration.")

import copy

model = build_model(encoder=CONFIG["encoder"], encoder_weights="imagenet",
                    arch=CONFIG["arch"], decoder_attention_type=CONFIG["decoder_attention_type"])
model = model.to(device=device, memory_format=torch.channels_last)

class ModelEMA:
    """Exponential moving average of weights — evaluated and exported."""
    def __init__(self, net, decay):
        self.module = copy.deepcopy(net).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
    @torch.no_grad()
    def update(self, net):
        src = net.state_dict()
        for k, v in self.module.state_dict().items():
            s = src[k].detach()
            if v.dtype.is_floating_point:
                v.mul_(self.decay).add_(s, alpha=1.0 - self.decay)
            else:
                v.copy_(s)

ema = ModelEMA(model, CONFIG["ema_decay"])

def optimizer_groups(net, base_lr, enc_scale, wd):
    groups = defaultdict(list)
    for name, p in net.named_parameters():
        if not p.requires_grad:
            continue
        branch = "encoder" if name.startswith("encoder.") else "decoder"
        decay = "no_decay" if p.ndim == 1 or name.endswith(".bias") else "decay"
        groups[(branch, decay)].append(p)
    out = []
    for (branch, decay), params in groups.items():
        out.append({"params": params,
                    "lr": base_lr * (enc_scale if branch == "encoder" else 1.0),
                    "weight_decay": 0.0 if decay == "no_decay" else wd})
    return out

optimizer = torch.optim.AdamW(
    optimizer_groups(model, CONFIG["base_lr"], CONFIG["encoder_lr_scale"], CONFIG["weight_decay"]),
    betas=(0.9, 0.999), eps=1e-8)
loss_fn = ComboLoss(bce_weight=CONFIG["bce_weight"], dice_weight=CONFIG["dice_weight"],
                    lovasz_weight=CONFIG["lovasz_weight"], cldice_weight=0.0)
scaler = torch.amp.GradScaler("cuda", enabled=True)

updates_per_epoch = math.ceil(len(train_loader) / CONFIG["grad_accum"])
total_updates = updates_per_epoch * CONFIG["epochs"]
warmup_updates = updates_per_epoch * CONFIG["warmup_epochs"]

def lr_factor(update):
    if update < warmup_updates:
        return max(1e-3, (update + 1) / max(1, warmup_updates))
    phase = (update - warmup_updates) / max(1, total_updates - warmup_updates)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, phase)))
    return CONFIG["min_lr_ratio"] + (1.0 - CONFIG["min_lr_ratio"]) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_factor)
print("model:", CONFIG["encoder"], "+", CONFIG["arch"], CONFIG["decoder_attention_type"],
      "| params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))
print("effective batch:", CONFIG["batch_size"] * CONFIG["grad_accum"],
      "| total optimizer updates:", total_updates)

In [ ]:
# 7. Train; update EMA each step; validate the EMA model at full resolution.
import csv
from tqdm.auto import tqdm

def unwrap(net):
    return net.module if hasattr(net, "module") else net

def save_state(path, epoch, best_iou, threshold, metrics, include_optimizer):
    payload = {
        "state_dict": ema.module.state_dict(),   # export the EMA weights
        "meta": {
            "encoder": CONFIG["encoder"], "arch": CONFIG["arch"],
            "decoder_attention_type": CONFIG["decoder_attention_type"],
            "image_size": CONFIG["image_size"], "threshold": float(threshold),
            "best_full_resolution_iou": float(best_iou), "epoch": int(epoch),
            "config": CONFIG.copy(), "metrics": metrics,
        },
    }
    if include_optimizer:
        payload.update({"raw_state_dict": unwrap(model).state_dict(),
                        "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                        "scaler": scaler.state_dict(), "epoch": int(epoch), "best_iou": float(best_iou)})
    torch.save(payload, path)

RESUME_PATH = None  # set to LAST_PATH to continue after a timed-out session
start_epoch, best_iou, best_threshold = 1, -1.0, 0.5
if RESUME_PATH:
    resumed = torch.load(RESUME_PATH, map_location="cpu", weights_only=False)
    unwrap(model).load_state_dict(resumed["raw_state_dict"])
    ema.module.load_state_dict(resumed["state_dict"])
    optimizer.load_state_dict(resumed["optimizer"]); scheduler.load_state_dict(resumed["scheduler"])
    scaler.load_state_dict(resumed["scaler"])
    start_epoch = int(resumed["epoch"]) + 1
    best_iou = float(resumed.get("best_iou", -1.0))
    best_threshold = float(resumed.get("meta", {}).get("threshold", 0.5))
    print(f"resumed from epoch {start_epoch - 1}; best IoU {best_iou:.5f}")

with open(HISTORY_PATH, "a", newline="") as hist:
    writer = csv.DictWriter(hist, fieldnames=["epoch", "train_loss", "lr", "cldice_weight", "val_iou", "threshold"])
    if hist.tell() == 0:
        writer.writeheader()

    for epoch in range(start_epoch, CONFIG["epochs"] + 1):
        # Switch on the connectivity term only after pixel segmentation is stable.
        loss_fn.cldice_weight = (CONFIG["cldice_weight"]
                                 if epoch > CONFIG["epochs"] - CONFIG["topology_epochs"] else 0.0)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running, batches = 0.0, 0
        progress = tqdm(train_loader, desc=f"epoch {epoch:02d}/{CONFIG['epochs']}", leave=False)
        for i, (images, masks) in enumerate(progress):
            group_start = i - (i % CONFIG["grad_accum"])
            group_size = min(CONFIG["grad_accum"], len(train_loader) - group_start)
            images = images.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
            masks = masks.to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", enabled=True):
                loss = loss_fn(model(images), masks)
            scaler.scale(loss / group_size).backward()
            running += float(loss.detach()); batches += 1
            if ((i + 1) % CONFIG["grad_accum"] == 0) or (i + 1 == len(train_loader)):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True); scheduler.step()
                ema.update(model)
            progress.set_postfix(loss=f"{running / max(batches, 1):.4f}",
                                 lr=f"{optimizer.param_groups[-1]['lr']:.2e}")

        row = {"epoch": epoch, "train_loss": running / max(batches, 1),
               "lr": optimizer.param_groups[-1]["lr"], "cldice_weight": loss_fn.cldice_weight,
               "val_iou": "", "threshold": ""}
        validate_now = epoch == 1 or epoch % CONFIG["validate_every"] == 0 or epoch == CONFIG["epochs"]
        metrics = {"validation_skipped": True}
        if validate_now:
            metrics = evaluate_full_images(ema.module, val_pairs, CONFIG["thresholds"],
                                           use_flip=CONFIG["val_flip_tta"])
            row["val_iou"], row["threshold"] = metrics["iou"], metrics["threshold"]
            print(f"epoch {epoch:02d} | loss {row['train_loss']:.4f} | EMA full-val IoU "
                  f"{metrics['iou']:.5f} @ thr {metrics['threshold']:.2f} | lr {row['lr']:.2e}")
            if metrics["iou"] > best_iou:
                best_iou, best_threshold = metrics["iou"], metrics["threshold"]
                save_state(BEST_PATH, epoch, best_iou, best_threshold, metrics, include_optimizer=False)
                print("  saved new best EMA checkpoint ->", BEST_PATH)
        else:
            print(f"epoch {epoch:02d} | loss {row['train_loss']:.4f} | validation deferred | lr {row['lr']:.2e}")

        writer.writerow(row); hist.flush()
        save_state(LAST_PATH, epoch, best_iou, best_threshold, metrics, include_optimizer=True)
        gc.collect(); torch.cuda.empty_cache()

assert BEST_PATH.exists(), "No best checkpoint was written. Inspect the training log above."
print(f"best EMA full-resolution IoU: {best_iou:.5f} @ threshold {best_threshold:.2f}")

In [ ]:
# 8. Load the best EMA weights, then the final flip + multi-scale TTA evaluation.
best_payload = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
eval_model = build_model(encoder=CONFIG["encoder"], encoder_weights=None,
                         arch=CONFIG["arch"], decoder_attention_type=CONFIG["decoder_attention_type"])
eval_model.load_state_dict(best_payload["state_dict"])
eval_model = eval_model.to(device=device, memory_format=torch.channels_last).eval()

final_metrics = evaluate_full_images(eval_model, val_pairs, CONFIG["thresholds"],
                                     use_flip=CONFIG["final_flip_tta"],
                                     scales=tuple(CONFIG["final_tta_scales"]))
final_threshold = final_metrics["threshold"]
print(f"final full-resolution flip+scale-TTA IoU: {final_metrics['iou']:.5f} @ thr {final_threshold:.2f}")

best_payload["meta"].update({
    "threshold": float(final_threshold), "final_flip_tta": bool(CONFIG["final_flip_tta"]),
    "final_tta_scales": list(CONFIG["final_tta_scales"]),
    "final_full_resolution_iou": float(final_metrics["iou"]), "final_metrics": final_metrics,
})
torch.save(best_payload, BEST_PATH)

In [ ]:
# 9. Occlusion-aware deployment threshold.
# Maximise hidden-road recovery while keeping clean IoU within a small budget of its peak.
MAX_ALLOWED_CLEAN_IOU_DROP = 0.010

def occlude_image(image, rng, max_holes=5):
    image = image.copy(); h, w = image.shape[:2]
    occ = np.zeros((h, w), np.uint8)
    for _ in range(rng.integers(1, max_holes + 1)):
        hh = int(rng.integers(max(8, S // 16), max(9, S // 7)))
        ww = int(rng.integers(max(8, S // 16), max(9, S // 7)))
        y = int(rng.integers(0, max(1, h - hh + 1))); x = int(rng.integers(0, max(1, w - ww + 1)))
        image[y:y + hh, x:x + ww] = 0; occ[y:y + hh, x:x + ww] = 1
    return image, occ

candidate_thresholds = sorted(float(t) for t in final_metrics["iou_by_threshold"])
clean_best_iou = float(final_metrics["iou"])
occ_rng = np.random.default_rng(SEED + 99)
hidden_road = 0
recovered = {t: 0 for t in candidate_thresholds}
N_OCC = min(100, len(val_pairs))
for image_path, mask_path in val_pairs[:N_OCC]:
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    target = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127
    occluded, occ_mask = occlude_image(image, occ_rng)
    prob = predict_prob(eval_model, occluded, use_flip=CONFIG["final_flip_tta"])  # flip-TTA, single scale
    hidden = target & occ_mask.astype(bool)
    hidden_road += hidden.sum(dtype=np.int64)
    for t in candidate_thresholds:
        recovered[t] += ((prob >= t) & hidden).sum(dtype=np.int64)

report = [{"threshold": t, "clean_iou": float(final_metrics["iou_by_threshold"][t]),
           "occlusion_recall": recovered[t] / max(hidden_road, 1)} for t in candidate_thresholds]
eligible = [r for r in report if r["clean_iou"] >= clean_best_iou - MAX_ALLOWED_CLEAN_IOU_DROP]
selected = max(eligible, key=lambda r: (r["occlusion_recall"], r["clean_iou"]))
final_threshold = float(selected["threshold"]); occ_recall = float(selected["occlusion_recall"])

print("threshold | clean IoU | occlusion recall | eligible")
for r in report:
    allowed = r["clean_iou"] >= clean_best_iou - MAX_ALLOWED_CLEAN_IOU_DROP
    flag = "<-- selected" if r["threshold"] == final_threshold else ""
    print(f"{r['threshold']:.2f}      | {r['clean_iou']:.5f} | {r['occlusion_recall']:.5f}        | {allowed} {flag}")
print(f"\nSelected threshold: {final_threshold:.2f} | clean IoU {selected['clean_iou']:.5f} | "
      f"Occlusion-Recall: {occ_recall:.5f}")

best_payload["meta"].update({
    "threshold": final_threshold,
    "threshold_selection": "max occlusion recall with <= 0.010 clean-IoU drop",
    "occlusion_recall": occ_recall, "clean_iou_at_selected_threshold": float(selected["clean_iou"]),
})
torch.save(best_payload, BEST_PATH)

In [ ]:
# 10. Visual sanity check and explicit export paths.
import matplotlib.pyplot as plt

image_path, mask_path = val_pairs[0]
image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
target = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127
prob = predict_prob(eval_model, image, use_flip=CONFIG["final_flip_tta"],
                    scales=tuple(CONFIG["final_tta_scales"]))
prediction = prob >= final_threshold

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(image); axes[0].set_title("satellite image")
axes[1].imshow(target, cmap="gray"); axes[1].set_title("ground truth")
axes[2].imshow(prob, cmap="magma", vmin=0, vmax=1); axes[2].set_title("road probability")
axes[3].imshow(prediction, cmap="gray"); axes[3].set_title(f"prediction @ {final_threshold:.2f}")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

print("Best deployable checkpoint:", BEST_PATH, "| exists:", BEST_PATH.exists())
print("Resumable checkpoint:", LAST_PATH, "| exists:", LAST_PATH.exists())
print("meta:", {k: best_payload["meta"][k] for k in
      ("encoder", "arch", "decoder_attention_type", "threshold",
       "final_full_resolution_iou", "occlusion_recall") if k in best_payload["meta"]})
print("\nKaggle: Save Version to retain /kaggle/working/road_segmentation as notebook output.")

## Inference notes

The exported checkpoint holds the **EMA** weights plus `meta` with `encoder`,
`arch`, `decoder_attention_type` and the occlusion-aware `threshold`, so the
project loader rebuilds and deploys it unchanged:

```python
from src.pipeline.p1_segment.model import load_checkpoint, predict_mask
model, meta = load_checkpoint("models/<this>.pt")   # rebuilds SCSE U-Net from meta
```

For city-scale AOIs keep the same 512 px / 384 px sliding-window policy used in
validation; enable flip (+ multi-scale) TTA only when accuracy matters more than
speed. The validation score is a quality signal, not a leaderboard number — test
on geographically separate imagery before making routing-quality claims.